<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/02%20-%20Supervised%20Learning/02b_validation_and_tuning.ipynb)

# 02b · 模型评估防坑：交叉验证、超参数调优与适用域

在上一节（`02a`）里，我们把数据按 8:2 随机切了一刀，算出随机森林的测试集 $\text{RMSE} = 1.17$，并在最后发现它在预测并五苯时出现了明显的“贴边”失真。

在实际做科研或审稿时，仅仅汇报一个单次划分的测试集分数是远远不够的。这一节我们要深究三个直接决定实验成败的问题：
1. **一次考试的成绩靠得住吗？**——换个随机数种子，分数会晃动多少？怎样用 **K 折交叉验证** 给出带误差棒的可靠评估？
2. **模型的“旋钮（超参数）”该怎么调？**——怎样用 **网格搜索（`GridSearchCV`）** 和 **流水线（`Pipeline`）** 在不发生数据泄漏的前提下选出最佳配置？再多测 1000 个实验数据值得吗（**学习曲线**）？
3. **面对一个全新的分子，模型的预测到底能不能信？**——当测试分子拥有全新骨架（**Scaffold Split**）或更大共轭尺寸（**外推划分**）时模型会怎样失效？如何用 **适用域（Applicability Domain）** 提前预警？

### 本节内容
1. 从单次划分的随机波动到 K 折交叉验证
2. 超参数网格搜索（`GridSearchCV`）与调参的真实收益
3. 用 `Pipeline` 杜绝交叉验证中的数据泄漏
4. 学习曲线（`learning_curve`）：评估数据量是否充足
5. 更严苛的泛化考验：骨架划分（Scaffold Split）与跨尺寸外推
6. 适用域（Applicability Domain）：量化预测的可信度

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "02 - Supervised Learning"))
    get_ipython().run_line_magic("pip", "install -q rdkit")


## 0. 环境准备与数据读入

我们先导入所需的库，并按与 `02a` 完全相同的流程读入清洗后的 AqSolDB 数据集（9480 个有效分子、17 个描述符）：

In [ ]:
try:
    import rdkit
except ImportError:
    %pip install -q rdkit
    import rdkit

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

RDLogger.DisableLog("rdApp.*")

def calc_rmse(y_true, y_pred):
    """计算均方根误差 (RMSE)。"""
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [ ]:
DATA_FILE = "data/solubility.csv"
if not os.path.exists(DATA_FILE):
    DATA_FILE = "solubility.csv"
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("请上传 solubility.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("找不到 solubility.csv，请确保它在 data/ 目录下")

df = pd.read_csv(DATA_FILE)
df["mol"] = df["SMILES"].apply(Chem.MolFromSmiles)
df = df[df["mol"].notna()].reset_index(drop=True)

feature_cols = [
    "MolWt", "MolLogP", "MolMR", "HeavyAtomCount", "NumHAcceptors", "NumHDonors",
    "NumHeteroatoms", "NumRotatableBonds", "NumValenceElectrons", "NumAromaticRings",
    "NumSaturatedRings", "NumAliphaticRings", "RingCount", "TPSA", "LabuteASA",
    "BalabanJ", "BertzCT"
]
X = df[feature_cols].values
y = df["Solubility"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"数据集准备完毕：训练集 {len(y_train)} 个分子，测试集 {len(y_test)} 个分子。")

---
## 1. 从单次划分的随机波动到 K 折交叉验证

### 1.1 换个随机种子，测试集分数会变多少？

在 `02a` 里，我们划分数据时随手写了 `random_state=42`，得到随机森林的测试 $\text{RMSE} = 1.17$。如果模型和超参数完全不动，仅仅把划分数据的随机种子换成 `0, 1, 2, 3, 4`（也就是重新洗牌抽 20% 的分子当测试集），测试 RMSE 会变多少？

In [ ]:
for seed in [0, 1, 2, 3, 4]:
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=seed)
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(Xa, ya)
    print(f"划分随机种子 random_state = {seed}  -->  测试集 RMSE = {calc_rmse(yb, model.predict(Xb)):.3f}")

可以看到，仅仅因为哪些难测的分子恰好被分进了测试集，完全相同的模型考出来的 RMSE 就在 **`1.10` 到 `1.18`** 之间上下波动（跨度达 `0.08`）！

这告诉我们一个非常重要的事实：**如果两个模型在单次测试集上的 RMSE 只差了 0.02 或 0.03，这完全可能是抽样运气造成的，不能据此声称一个算法比另一个更好。**

### 1.2 K 折交叉验证（K-Fold Cross-Validation）

为了消除单次抽样的偶然性，同时不偷看锁在抽屉里的最终测试集 `X_test`，标准做法是在**训练集 `X_train` 内部**进行 **K 折交叉验证**（最常用 $K=5$）：

```text
把训练集均分为 5 份，轮流拿 1 份当验证集（Validation）、其余 4 份当训练集：
第 1 轮:  [ 验证 ] [ 训练 ] [ 训练 ] [ 训练 ] [ 训练 ]  -->  得分 1
第 2 轮:  [ 训练 ] [ 验证 ] [ 训练 ] [ 训练 ] [ 训练 ]  -->  得分 2
第 3 轮:  [ 训练 ] [ 训练 ] [ 验证 ] [ 训练 ] [ 训练 ]  -->  得分 3
第 4 轮:  [ 训练 ] [ 训练 ] [ 训练 ] [ 验证 ] [ 训练 ]  -->  得分 4
第 5 轮:  [ 训练 ] [ 训练 ] [ 训练 ] [ 训练 ] [ 验证 ]  -->  得分 5
```

5 轮下来，训练集里的每一个分子都恰好当过一次“模拟考题”。最后我们汇报这 5 个分数的**平均值 ± 标准差**：

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    X_train, y_train,
    cv=cv,
    scoring="neg_root_mean_squared_error",   # sklearn 内部约定“分数越大越好”，因此误差指标前面带负号
)

cv_rmse = -scores   # 去掉负号还原为正的 RMSE
print("5 折各自的验证 RMSE:", cv_rmse.round(3))
print(f"5 折交叉验证结果: RMSE = {cv_rmse.mean():.3f} ± {cv_rmse.std():.3f}")

在论文或报告中写出 `RMSE = 1.142 ± 0.036 (5-fold CV)`，既交代了平均水平，又交代了不确定度，比孤零零报一个数字严谨得多。

---
## 2. 超参数网格搜索（`GridSearchCV`）

在机器学习里，有两种“参数”初学者经常混淆：

| 概念 | 由谁决定 | 典型例子 |
|---|---|---|
| **模型参数（Parameters）** | 调用 `fit()` 时由算法**从数据中自动学出** | 线性回归的权重 $w_i$、每棵决策树在哪个特征的哪个阈值分叉 |
| **超参数（Hyperparameters）** | 在训练开始**之前**，由我们**人工设定**的模型旋钮 | 随机森林的树数 `n_estimators`、最大树深 `max_depth`、kNN 的邻居数 `n_neighbors` |

在 `02a` 里我们发现，默认不限深度（`max_depth=None`）的随机森林在训练集上 $\text{RMSE} = 0.42$、测试集上 $1.17$，存在明显过拟合。如果我们限制树的最大深度 `max_depth`，或者把树的数量从 100 棵增加到 300 棵，能不能降低过拟合、提升泛化能力？

**网格搜索（`GridSearchCV`）** 会把我们列出的候选超参数组合成一张网格（下面共 $2 \times 4 = 8$ 种组合），对每一种组合自动跑一遍 5 折交叉验证，挑出平均验证误差最小的那一组（总共训练 $8 \times 5 = 40$ 个随机森林，约需半分钟）：

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [5, 10, 20, None],   # None 代表不限制深度，一直分裂到叶子节点纯净为止
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
)
grid.fit(X_train, y_train)   # 铁律：调参只能在训练集 X_train 上进行，绝不能碰 X_test！

print("最优超参数组合:", grid.best_params_)
print(f"最优 5 折交叉验证 RMSE: {-grid.best_score_:.3f}")

> **为什么 `grid.fit()` 绝不能把 `X_test` 放进去？**
> 如果看着测试集 `X_test` 的分数来挑选超参数，就相当于“拿着期末考试真题去挑选复习策略”，挑出来的参数只是恰好迎合了这批测试分子，测试集也就失去了独立考评的意义。这同样属于**数据泄漏**。

我们把这 8 种超参数组合的 5 折交叉验证成绩展开成表格看看：

In [ ]:
cv_table = pd.DataFrame(grid.cv_results_)
cv_table["CV_RMSE"] = -cv_table["mean_test_score"]
cv_table[["param_max_depth", "param_n_estimators", "CV_RMSE", "std_test_score"]].round(3)

仔细看上面这张表，有两个非常接地气的实战结论：
1. **树太浅会导致欠拟合**：当强行把树深限制为 `max_depth=5` 时，虽然树没法死记硬背训练数据了，但因为分支太少、表达能力不足，验证集 RMSE 反而恶化到了 `1.275`。
2. **调参不是魔法**：当深度放宽到 `20` 或 `None`（不限深度）、树数从 `100` 增加到 `300` 时，验证集 RMSE 在 `1.136 ~ 1.142` 之间（`max_depth=20` 与 `None` 差距不到 `0.001`）。也就是说，**花了 40 倍算力做网格搜索，带来的提升只有约 `0.005`——还不到第 1 节换一个随机种子带来的波动（`0.08`）的十分之一！**

`GridSearchCV` 在选出最优组合后，默认会自动用这组最优参数在**整个 `X_train`** 上重新训练一遍最终模型，保存在 `grid.best_estimator_` 中：


In [ ]:
rf_gs = grid.best_estimator_
print(f"调参后的最优模型在测试集上的 RMSE = {calc_rmse(y_test, rf_gs.predict(X_test)):.3f}")

### 🏋️ 练习 02b-1

训练一个默认参数的随机森林 `rf_default = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train, y_train)`，用 `calc_rmse()` 分别计算 `rf_default` 和上面的 `rf_gs` 在**训练集**与**测试集**上的 RMSE。对比看看：
1. 调参后的模型测试集 RMSE 改善了多少？对照第 1 节的波动幅度，这个差距显著吗？
2. 调参后的模型，训练 RMSE 与测试 RMSE 之间的差距（过拟合）消失了吗？

In [ ]:
# 在这里写你的代码


---
## 3. 用 `Pipeline` 杜绝交叉验证中的数据泄漏

对于随机森林，我们不需要缩放特征，直接调用 `cross_val_score` 或 `GridSearchCV` 就行。但如果是 `02a` 里必须做 `StandardScaler` 缩放的 **kNN** 或 **SVR** 呢？

如果在做 5 折交叉验证**之前**，先写了一句 `X_train_s = StandardScaler().fit_transform(X_train)`，然后把 `X_train_s` 传给 `cross_val_score`，就又犯了隐蔽的**数据泄漏**：因为在第 1 折验证时，缩放器里的均值和标准差已经包含了第 1 折验证集分子的信息！

正确的做法是：**在 5 折交叉验证的每一轮内部，只拿当前那 4 份训练折去 `fit` 缩放器，再分别 `transform` 那 4 份训练折和 1 份验证折。**

难道要我们自己手写循环去做这件繁琐的事吗？不用！`scikit-learn` 的 **`Pipeline`（流水线，`make_pipeline`）** 可以把“特征缩放器 + 回归模型”捆绑成一个整体对象。你把它交给 `cross_val_score` 或 `GridSearchCV` 时，它会在每一折内部自动按规矩独立执行缩放和训练：

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

knn_pipe = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5))

knn_scores = -cross_val_score(knn_pipe, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")
print(f"kNN 流水线 (缩放 + k=5) 5 折交叉验证 RMSE: {knn_scores.mean():.3f} ± {knn_scores.std():.3f}")

当需要对 `Pipeline` 里的某一步做 `GridSearchCV` 超参数搜索时，参数字典里的键名写作 **`步骤名称__参数名`**（注意中间是**两个下划线 `__`**）。`make_pipeline` 会自动把步骤命名为类名的小写形式，例如 `KNeighborsRegressor` 对应的步骤名就是 `kneighborsregressor`。

### 🏋️ 练习 02b-2

虽然随机森林对调参不太敏感，但 kNN 对邻居数 $k$ 和投票权重方式非常敏感。请用 `GridSearchCV` 为 `knn_pipe` 搜索以下两个超参数的最佳组合（5 折交叉验证）：
- `"kneighborsregressor__n_neighbors"`: `[3, 5, 10, 20, 40]`
- `"kneighborsregressor__weights"`: `["uniform", "distance"]`（`"uniform"` 表示 $k$ 个邻居平权平均；`"distance"` 表示距离越近的邻居话语权越大）

打印出选出的最优参数组合 `best_params_`、交叉验证 RMSE，以及该最优流水线在测试集 `X_test` 上的 RMSE。

In [ ]:
# 在这里写你的代码


---
## 4. 学习曲线：再多测 1000 个分子值得吗？

在真实的材料研发项目里，导师或工业界合作伙伴最常问的一个问题是：**“我们现在测了 500 个样品，模型误差还是有点大。如果再花半年时间多合成、多测 1000 个样品，模型到底能不能明显变好？”**

回答这个问题的标准工具是**学习曲线（`learning_curve`）**：我们人为地只抽取训练集的 5%（约 380 个分子）、10%、25%、50%、100%（7584 个分子）来训练模型，画出训练误差和交叉验证误差随样本量增长的变化趋势：

In [ ]:
from sklearn.model_selection import learning_curve

sizes, train_scores, val_scores = learning_curve(
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    X_train, y_train,
    train_sizes=[0.05, 0.1, 0.25, 0.5, 1.0],
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

fig, ax = plt.subplots(figsize=(5.6, 3.8))
ax.plot(sizes, -train_scores.mean(axis=1), "o-", label="Training RMSE")
ax.plot(sizes, -val_scores.mean(axis=1), "s-", label="5-fold CV RMSE")
ax.set_xscale("log")
ax.set_xlabel("Number of training molecules (log scale)")
ax.set_ylabel("RMSE (logS units)")
ax.set_title("Learning curve (Random Forest)")
ax.legend()
plt.tight_layout()
plt.show()

怎样从学习曲线上做科研决策？
- **看橙色验证曲线的斜率**：从 380 个分子（$\text{RMSE} \approx 1.38$）增加到 7584 个分子（$\text{RMSE} \approx 1.15$），验证误差一直在稳步下降（在对数横轴上几乎是一条向下倾斜的直线），说明**当前模型还没有“吃饱”，继续补充高质量实验数据仍然能降低误差**。
- 反之，如果橙色曲线早就走平了，而且和蓝色训练曲线都停在一个很高的误差上（比如 1.8），那就说明**瓶颈在于特征信息量不够或模型太简单**——此时再花钱测一万个数据也是白费力气，应该去补充关键物理特征（比如后面要讲的熔点或量子化学描述符）。

---
## 5. 更严苛的泛化考验：骨架划分与尺寸外推

前面无论是 `train_test_split` 还是 `KFold`，用的都是**随机划分（Random Split）**。但在化学数据集里，随机划分往往会让模型沾便宜：比如数据库里有一组同系物（邻氯苯胺、间氯苯胺、对氯苯胺），随机洗牌后，前两个进了训练集，第三个进了测试集。模型只要记住“氯苯胺的 $\log S$ 大概是 $-2$”就能拿高分，但这并不代表它具备预测**全新分子系列**的能力。

为了真实检验模型在探索新化学空间时的泛化表现，我们需要设计更贴近科研实战的划分方式。

### 5.1 按 Murcko 骨架划分（Scaffold Split）

还记得 `01d` 第 4.3 节学的 **Bemis–Murcko 骨架**吗？我们可以先提取出每个分子的核心骨架，然后用 `GroupShuffleSplit` 强制规定：**凡是具有相同 Murcko 骨架的分子，要么全部进训练集，要么全部进测试集，绝不允许跨界！** 这样测试集里的每一个骨架，模型在训练时都从未见过：

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

scaffolds = df["mol"].apply(lambda m: MurckoScaffold.MurckoScaffoldSmiles(mol=m))
groups = scaffolds.replace("", "acyclic")   # 无环脂肪族分子的骨架为空字符串，统一归为 "acyclic" 组

print("全库包含的不同 Murcko 骨架总数:", groups.nunique())
print("\n占比最高的前 5 种骨架（百分比 %）:")
print((groups.value_counts(normalize=True).head(5) * 100).round(1))

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

rf_scaf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_scaf.fit(X[train_idx], y[train_idx])

print("随机划分 (Random Split)   测试集 RMSE = 1.17")
print(f"骨架划分 (Scaffold Split) 测试集 RMSE = {calc_rmse(y[test_idx], rf_scaf.predict(X[test_idx])):.2f}")

切换到更严格的骨架划分后，测试集 RMSE 从 `1.17` 略微上升到了 `1.20`。为什么在 AqSolDB 上差别不算剧烈？
看上面打印出来的骨架占比就明白了：AqSolDB 是一个通用小分子库，其中 **29.7% 的分子完全没有环（`acyclic`）**，另有 **18.2% 的分子核心骨架就只是一个单苯环（`c1ccccc1`）**；而且模型输入的是 `MolLogP`、`MolWt` 这类通用物理化学描述符，在不同小分子骨架之间本来就有较好的可迁移性。如果在专门针对某几类复杂杂环骨架的药物或光电材料数据集上（尤其当使用分子指纹建模时），Scaffold Split 的误差往往会比随机划分高出一大截。

### 5.2 跨尺寸外推考验（Extrapolation Split）

比“换一种同尺寸骨架”更残酷的考验是**尺寸外推**：在有机电子材料研发中，公开数据库里绝大多数是 1~2 个环的小分子，而我们想设计合成的往往是 4 个芳香环以上的大共轭体系。

我们直接模拟这个场景：
- **训练集**：只给模型看芳香环数 **$\le 2$** 的分子（共 8647 个）；
- **测试集**：让模型去预测芳香环数 **$\ge 4$** 的大共轭分子（共 261 个）。

看看**线性回归**和**随机森林**在这场跨尺寸外推大考中会交出怎样的答卷：

In [ ]:
n_arom = df["NumAromaticRings"].values
small_mask = n_arom <= 2
large_mask = n_arom >= 4
print(f"训练集 (<=2 个芳香环): {small_mask.sum()} 个 | 外推测试集 (>=4 个芳香环): {large_mask.sum()} 个\n")

extrap_preds = {}
for name, model in [
    ("Linear regression", LinearRegression()),
    ("Random forest", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
]:
    model.fit(X[small_mask], y[small_mask])
    pred = model.predict(X[large_mask])
    extrap_preds[name] = pred
    print(f"{name:18s} | 外推 RMSE = {calc_rmse(y[large_mask], pred):.2f} | 预测出的最低 logS = {pred.min():.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.8, 4.4))
for ax, (name, pred) in zip(axes, extrap_preds.items()):
    ax.scatter(y[large_mask], pred, s=12, alpha=0.55)
    ax.plot([-14, 2], [-14, 2], "k--", lw=1, label="Ideal (y = x)")
    ax.set_xlabel("Measured logS (log mol/L)")
    ax.set_ylabel("Predicted logS (log mol/L)")
    ax.set_title(f"{name}\n(Train: <=2 rings, Test: >=4 rings)")
    ax.legend()
plt.tight_layout()
plt.show()

上面这两张图堪称机器学习在材料科学中的经典教材案例——**两种模型在外推时以截然相反的方式崩溃了**：
- **左图（线性回归）：直线飞出天际**。因为线性公式一旦超出训练范围就会顺着斜率无止境地延伸，加上重原子数与价电子数等共线特征在大分子上比例失调，它竟然把某些大分子预测成了 $\log S = -46.5$（相当于在整个地球的海洋里都找不到一个溶质分子），外推 RMSE 暴涨到 **`5.28`**！
- **右图（随机森林）：全员卡在天花板/地板上**。正如我们在 `02a` 结尾分析的，树模型只能输出训练样本的局部平均值，绝不会跑出训练集的范围。但正因为训练集里根本没有 4 环以上的大分子，所有真正极难溶的大分子（真实 $\log S$ 在 $-8 \sim -13$）全都被随机森林高估，齐刷刷地“托底”卡在了 $-6 \sim -8$ 的水平线上（系统性低估了大共轭核带来的溶解度下降）。

> **这对材料发现意味着什么？**
> 在材料研发中，我们最渴望找到的往往是“性质超越现有数据库纪录”的新材料（比如迁移率更高、带隙更窄、或者催化活性更强）。但要时刻清醒：**纯数据驱动的树模型是无法帮你预测出超越训练集上限的极端值的**；而在外推区域，必须结合物理方程约束或主动学习/贝叶斯优化（第 4 章内容）来逐步探索。

---
## 6. 适用域（Applicability Domain）：这个预测能信吗？

既然模型只在“跟训练数据长得像”的分子上才靠谱，那么当实验同事拿来一个新分子让你预测时，你怎样定量告诉他：“这个分子在咱们模型的能力圈内，预测可信”还是“这个分子离训练集太远了，预测纯属瞎蒙”？

这个“能力圈”在化学信息学里叫做模型的**适用域（Applicability Domain, AD）**。最直观的衡量办法就是利用 `01d` 学的 **Morgan 指纹与 Tanimoto 相似度**：计算新分子与整个训练集所有分子的 Tanimoto 相似度，取其中的最大值（即**最近邻相似度 `max_similarity`**）。

我们先在测试集 `idx_test` 上验证一下：离训练集越远的分子，随机森林的预测误差是不是真的越大？

In [ ]:
fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# 获取与 02a 完全一致的训练集、测试集行下标
idx_train, idx_test = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42)
train_fps = [fp_gen.GetFingerprint(df["mol"][i]) for i in idx_train]

def get_max_similarity(mol, ref_fps):
    """返回分子 mol 与参考指纹列表 ref_fps 中最相似分子的 Tanimoto 相似度。"""
    return max(DataStructs.BulkTanimotoSimilarity(fp_gen.GetFingerprint(mol), ref_fps))

test_max_sim = np.array([get_max_similarity(df["mol"][i], train_fps) for i in idx_test])

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X[idx_train], y[idx_train])
test_sq_err = (rf.predict(X[idx_test]) - y[idx_test]) ** 2

# 按最近邻相似度分成 4 档，分别统计样本数与 RMSE：
sim_bins = pd.cut(test_max_sim, bins=[0.0, 0.3, 0.5, 0.7, 1.0], include_lowest=True)
ad_summary = pd.DataFrame({"sim_bin": sim_bins, "sq_err": test_sq_err}).groupby("sim_bin", observed=True)["sq_err"]
pd.DataFrame({"分子个数": ad_summary.size(), "测试 RMSE": np.sqrt(ad_summary.mean())}).round(2)

结果一目了然：当测试分子与训练集的最大相似度在 `0.3` 以上时，RMSE 稳定在 `1.0 ~ 1.2` 左右；而一旦最大相似度跌到 **`0.3` 以下**（训练集里根本没有结构相近的参考物），RMSE 立刻恶化到 **`2.21`**——跟什么都不学的“瞎猜平均值基线（`2.42`）”几乎没区别了！

现在，我们把平时关心的 7 个代表性**有机半导体分子**拿过来，同时用两把尺子检查它们有没有跑出训练集的范围：
1. **结构相似度尺子**：与训练集分子的最大 Tanimoto 相似度，以及训练集里跟它最像的是哪个分子；
2. **描述符范围尺子**：它的疏水常数 `MolLogP` 排在训练集什么位置（训练集里还有百分之多少的分子比它更亲油）。

In [ ]:
osc_candidates = {
    "pentacene": "c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1",
    "3-hexylthiophene": "CCCCCCc1ccsc1",
    "BTBT": "c1ccc2c(c1)sc1c2sc2ccccc21",
    "C8-BTBT": "CCCCCCCCc1ccc2c(c1)sc1c3ccc(CCCCCCCC)cc3sc21",
    "rubrene": "c1ccc(cc1)-c1c2ccccc2c(-c2ccccc2)c2c(-c3ccccc3)c3ccccc3c(-c3ccccc3)c12",
    "TIPS-pentacene": "CC(C)[Si](C#Cc1c2cc3ccccc3cc2c(C#C[Si](C(C)C)(C(C)C)C(C)C)c2cc3ccccc3cc12)(C(C)C)C(C)C",
    "EDOT": "C1COc2cscc2O1",
}

train_logP = df["MolLogP"].values[idx_train]
ad_rows = []

for name, smi in osc_candidates.items():
    mol = Chem.MolFromSmiles(smi)
    sims = DataStructs.BulkTanimotoSimilarity(fp_gen.GetFingerprint(mol), train_fps)
    best_j = int(np.argmax(sims))
    logp_val = Descriptors.MolLogP(mol)
    ad_rows.append({
        "molecule": name,
        "max_similarity (bit FP)": round(max(sims), 2),
        "nearest_in_train": df["Name"][idx_train[best_j]],
        "MolLogP": round(logp_val, 1),
        "% train with higher logP": round(100 * (train_logP > logp_val).mean(), 2),
    })

pd.DataFrame(ad_rows)

看看这张适用域体检表，信息量非常丰富：
- **`TIPS-pentacene` 和 `EDOT`**：最大 Tanimoto 相似度连 `0.40` 都不到，说明训练集里根本没有含三异丙基硅乙炔基或二氧乙撑噻吩结构的类似物；
- **`C8-BTBT`、`rubrene`（红荧烯）、`TIPS-pentacene`**：它们的 `MolLogP` 高达 $9.6 \sim 12.6$，训练集里超过 99.6% 的分子都没它们这么疏水——它们已经完全处在描述符空间的极端边缘！
- **最具有欺骗性的是第一行 `pentacene`（并五苯）**：表里显示它与训练集中 `naphthacene`（并四苯的旧称）的最大相似度竟然是 **`1.00`**！难道并五苯完全在适用域内、预测绝对可靠吗？

当然不是！我们在 `02a` 结尾明明看到随机森林把并五苯预测错了。之所以这里算出来相似度是 `1.00`，正是因为我们用了**二值 Morgan 指纹（Bit FP）**——它在 `01d` 里就被我们抓到过分不清并四苯和并五苯！**用来衡量适用域的“尺子”如果选得不对，就会给出虚假的安全感。**

### 🏋️ 练习 02b-3

让我们换一把更准确的尺子：改用**计数指纹** `fp_gen.GetCountFingerprint(mol)` 重新生成训练集的指纹列表 `train_count_fps`，并计算并五苯（`pentacene`）与训练集所有分子的 Tanimoto 相似度（`DataStructs.BulkTanimotoSimilarity` 同样直接支持计数指纹）。
1. 换用计数指纹后，并五苯与训练集的最大相似度从 `1.00` 降到了多少？
2. 打印出与并五苯最相似的前 3 个训练集分子的名称和相似度。

In [ ]:
# 在这里写你的代码


---
## 小结

这一节我们系统学习了怎样科学、诚实地评估和诊断一个机器学习模型：
- **交叉验证（`KFold` / `cross_val_score`）**：单次 `train_test_split` 的分数会随随机种子显著波动（$\pm 0.04$），在训练集上做 5 折交叉验证并汇报 `均值 ± 标准差` 才能区分真实改进与随机运气；
- **超参数调优与防泄漏流水线（`GridSearchCV` + `Pipeline`）**：用 `make_pipeline` 把 `StandardScaler` 与模型捆绑，确保每一折验证都在训练折内部独立缩放；同时认识到当模型默认设置合理时，调参的边际收益往往远小于特征质量和数据量的影响；
- **学习曲线（`learning_curve`）**：通过观察验证误差随样本量增加是否继续下降，判断“继续补测实验数据”是否划算；
- **严苛划分与适用域（Scaffold Split / Extrapolation / AD）**：
  - 面对超出训练范围的大分子，线性模型会沿斜率飞出物理边界，而树模型则会卡在训练集极值处停止外推；
  - 在对新分子下结论前，务必先检查它与训练集的**最大结构相似度**（注意优先用计数指纹而非二值指纹）以及**核心描述符是否越界**。

**下一节（`02c_classification.ipynb`）**：我们将把连续数值预测转为二元分类任务（筛查“难溶分子”），探讨为什么在类别不平衡时“90% 的准确率”可能是个骗局，并通过特征重要性分析打开随机森林的黑箱。